# Semantic Similar-Ticket Retrieval

This notebook builds and evaluates a semantic retrieval system for
ResolveAI.

The objective is to retrieve historically similar customer-support
tickets for a new customer query using dense sentence embeddings.

The retrieval corpus contains only the leakage-safe training split.
Validation, challenge, and official test examples are excluded from
the retrieval index and are used only as retrieval queries.

This component will later provide contextual support examples for the
ResolveAI RAG resolution-generation pipeline.

In [1]:
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
SEED = 42

In [3]:
train_df = pd.read_csv(
    "../data/processed/generic/train.csv"
)

validation_df = pd.read_csv(
    "../data/processed/generic/validation.csv"
)

print("Training corpus:", train_df.shape)
print("Validation queries:", validation_df.shape)

Training corpus: (18780, 9)
Validation queries: (4016, 9)


In [4]:
retrieval_corpus = (
    train_df[
        [
            "processed_instruction",
            "intent",
            "response"
        ]
    ]
    .copy()
)

validation_queries = (
    validation_df[
        [
            "processed_instruction",
            "intent"
        ]
    ]
    .copy()
)

### Load CPU Embedding Model

In [5]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

Embedding dimension: 384


C:\Users\ribhu\AppData\Local\Temp\ipykernel_35444\289703839.py:3: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


### Encoding a Query

In [7]:
sample_text = (
    retrieval_corpus[
        "processed_instruction"
    ]
    .iloc[0]
)

sample_embedding = (
    embedding_model.encode(
        sample_text
    )
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    sample_embedding[:10]
)

Embedding shape: (384,)
[-0.03027715  0.05177317  0.10175335  0.01067996 -0.02903386 -0.00467919
  0.03092369 -0.05523818  0.10852908  0.09587106]


### Test semantic similarity manually

In [8]:
example_sentences = [
    "where is my refund",
    "the money from my return has not arrived",
    "how can I change my account email",
    "my refund is still missing"
]

example_embeddings = (
    embedding_model.encode(
        example_sentences,
        normalize_embeddings=True
    )
)

similarities = cosine_similarity(
    [example_embeddings[0]],
    example_embeddings
)[0]

for sentence, similarity in zip(
    example_sentences,
    similarities
):
    print(
        f"{similarity:.4f}  {sentence}"
    )

1.0000  where is my refund
0.5982  the money from my return has not arrived
0.1961  how can I change my account email
0.8247  my refund is still missing


### Encoding the full training corpus

In [9]:
corpus_texts = (
    retrieval_corpus[
        "processed_instruction"
    ]
    .tolist()
)

corpus_embeddings = (
    embedding_model.encode(
        corpus_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
)

print(
    "Corpus embedding shape:",
    corpus_embeddings.shape
)

Batches:   0%|          | 0/294 [00:00<?, ?it/s]

Corpus embedding shape: (18780, 384)


### Building the first retrieval function

In [10]:
def retrieve_similar_tickets(
    query,
    top_k=5
):

    query_embedding = (
        embedding_model.encode(
            [query],
            normalize_embeddings=True
        )
    )

    similarities = (
        cosine_similarity(
            query_embedding,
            corpus_embeddings
        )[0]
    )

    top_indices = (
        np.argsort(
            similarities
        )[::-1][:top_k]
    )

    results = (
        retrieval_corpus
        .iloc[top_indices]
        .copy()
    )

    results["similarity"] = (
        similarities[top_indices]
    )

    return results

In [11]:
retrieve_similar_tickets(
    "my refund still hasn't reached my bank account",
    top_k=5
)[
    [
        "processed_instruction",
        "intent",
        "similarity"
    ]
]

,processed_instruction,intent,similarity
11299,want help obtaining a refund of money,get_refund,0.743428
11432,getting refund of money,get_refund,0.740152
18408,I want help seeing if there is anything wrong ...,track_refund,0.738646
11669,how can i get a refund,get_refund,0.738150
11405,need assistance getting a refund of my money,get_refund,0.733610


### Encode validation queries

In [12]:
validation_texts = (
    validation_queries[
        "processed_instruction"
    ]
    .tolist()
)

validation_embeddings = (
    embedding_model.encode(
        validation_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
)

print(
    "Validation embedding shape:",
    validation_embeddings.shape
)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Validation embedding shape: (4016, 384)


### Creating a retrieval evaluator

In [13]:
def evaluate_retrieval(
    query_embeddings,
    query_intents,
    corpus_embeddings,
    corpus_intents,
    top_k=5,
    batch_size=128
):
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0

    reciprocal_ranks = []
    precision_at_5 = []

    corpus_intents = np.array(
        corpus_intents
    )

    query_intents = np.array(
        query_intents
    )

    for start in range(
        0,
        len(query_embeddings),
        batch_size
    ):

        end = start + batch_size

        batch_embeddings = (
            query_embeddings[start:end]
        )

        similarities = (
            batch_embeddings
            @ corpus_embeddings.T
        )

        top_indices = np.argsort(
            similarities,
            axis=1
        )[:, ::-1][:, :top_k]

        for row_idx, retrieved_indices in enumerate(
            top_indices
        ):
            query_index = start + row_idx

            true_intent = (
                query_intents[
                    query_index
                ]
            )

            retrieved_intents = (
                corpus_intents[
                    retrieved_indices
                ]
            )

            relevant = (
                retrieved_intents
                == true_intent
            )

            if relevant[:1].any():
                hits_at_1 += 1

            if relevant[:3].any():
                hits_at_3 += 1

            if relevant[:5].any():
                hits_at_5 += 1

            relevant_positions = np.where(
                relevant
            )[0]

            if len(relevant_positions) > 0:
                first_rank = (
                    relevant_positions[0]
                    + 1
                )

                reciprocal_ranks.append(
                    1 / first_rank
                )
            else:
                reciprocal_ranks.append(
                    0
                )

            precision_at_5.append(
                relevant[:5].mean()
            )

    num_queries = len(
        query_embeddings
    )

    return {
        "hit@1": (
            hits_at_1
            / num_queries
        ),
        "hit@3": (
            hits_at_3
            / num_queries
        ),
        "hit@5": (
            hits_at_5
            / num_queries
        ),
        "mrr@5": np.mean(
            reciprocal_ranks
        ),
        "precision@5": np.mean(
            precision_at_5
        )
    }

In [14]:
validation_retrieval_metrics = (
    evaluate_retrieval(
        query_embeddings=validation_embeddings,
        query_intents=validation_queries[
            "intent"
        ].tolist(),

        corpus_embeddings=corpus_embeddings,
        corpus_intents=retrieval_corpus[
            "intent"
        ].tolist(),

        top_k=5
    )
)

validation_retrieval_metrics

{'hit@1': 0.9868027888446215,
 'hit@3': 0.9930278884462151,
 'hit@5': 0.9942729083665338,
 'mrr@5': np.float64(0.9899236387782204),
 'precision@5': np.float64(0.9812749003984064)}

### Initial Semantic Retrieval Observations

The `all-MiniLM-L6-v2` embedding model produced sensible semantic
similarities in manual tests. Refund-related paraphrases received
substantially higher cosine similarity than an unrelated account-editing
query.

The training corpus was encoded into 18,780 normalized 384-dimensional
sentence embeddings.

Initial nearest-neighbor retrieval successfully identified the general
refund topic. However, a query describing an outstanding refund retrieved
mostly `get_refund` examples rather than `track_refund`, indicating that
dense semantic similarity may still struggle with fine-grained intent
boundaries.

A systematic retrieval evaluation is therefore required rather than
relying on qualitative examples alone.

## Per-Intent Retrieval Analysis

In [15]:
def build_retrieval_results(
    query_embeddings,
    query_intents,
    corpus_embeddings,
    corpus_intents,
    top_k=5,
    batch_size=128
):
    rows = []

    query_intents = np.array(
        query_intents
    )

    corpus_intents = np.array(
        corpus_intents
    )

    for start in range(
        0,
        len(query_embeddings),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(query_embeddings)
        )

        batch_embeddings = (
            query_embeddings[start:end]
        )

        # Because embeddings are normalized,
        # dot product == cosine similarity
        similarities = (
            batch_embeddings
            @ corpus_embeddings.T
        )

        top_indices = (
            np.argsort(
                similarities,
                axis=1
            )[:, ::-1][:, :top_k]
        )

        for row_idx, retrieved_indices in enumerate(
            top_indices
        ):
            query_index = (
                start + row_idx
            )

            true_intent = (
                query_intents[
                    query_index
                ]
            )

            retrieved_intents = (
                corpus_intents[
                    retrieved_indices
                ]
            )

            relevant = (
                retrieved_intents
                == true_intent
            )

            relevant_positions = (
                np.where(relevant)[0]
            )

            if len(relevant_positions) > 0:
                reciprocal_rank = (
                    1
                    / (
                        relevant_positions[0]
                        + 1
                    )
                )
            else:
                reciprocal_rank = 0.0

            rows.append({
                "query_index": query_index,
                "intent": true_intent,

                "top1_intent":
                    retrieved_intents[0],

                "hit@1":
                    int(relevant[:1].any()),

                "hit@3":
                    int(relevant[:3].any()),

                "hit@5":
                    int(relevant[:5].any()),

                "mrr@5":
                    reciprocal_rank,

                "precision@5":
                    relevant[:5].mean(),
            })

    return pd.DataFrame(rows)

In [16]:
validation_retrieval_results = (
    build_retrieval_results(
        query_embeddings=validation_embeddings,

        query_intents=validation_queries[
            "intent"
        ].tolist(),

        corpus_embeddings=corpus_embeddings,

        corpus_intents=retrieval_corpus[
            "intent"
        ].tolist(),

        top_k=5
    )
)

In [17]:
validation_retrieval_results.head()

,query_index,intent,top1_intent,hit@1,hit@3,hit@5,mrr@5,precision@5
0,0,cancel_order,cancel_order,1,1,1,1.0,1.0
1,1,cancel_order,cancel_order,1,1,1,1.0,1.0
2,2,cancel_order,cancel_order,1,1,1,1.0,1.0
3,3,cancel_order,cancel_order,1,1,1,1.0,1.0
4,4,cancel_order,cancel_order,1,1,1,1.0,1.0
